# Track B — Pakistani-Urdu cascade on Kaggle

**The safe track.** Full-duplex *feel* from three off-the-shelf parts:

```
mic --> streaming ASR --> Urdu LLM --> streaming TTS --> speaker
          ^------------- VAD + barge-in ----------------^
```

Not truly simultaneous like Moshi. But keep a VAD live while the bot talks,
kill its audio the instant the caller cuts in, and within ~half a second it
feels duplex. That barge-in *policy* is the hard part, so we test it in
isolation as a synchronous state machine.

**What runs where**

- Data prep, WER scoring, the orchestrator demo: **free Kaggle (CPU)**. All in
  this notebook, no GPU.
- A live call (real Whisper + LLM + Urdu TTS): **needs a GPU**. Marked clearly.

Acceptance criteria this track targets: **H3** (barge-in works), **H4** (stop
<= 500 ms), **H5** (response <= 1000 ms), **H6** (Urdu WER <= ~30%).

## 1. Setup

Install the package from the repo with the `[moshi]` extra (SentencePiece).
On Kaggle, add the repo as a dataset or clone it, then `pip install -e`.

In [ ]:
# >>> run once. On Kaggle the repo usually lives under /kaggle/input or you clone it.
# If you cloned it elsewhere, point REPO at that path.
import os, subprocess, sys

REPO = os.environ.get('DUPLEX_BOL_REPO', '/kaggle/working/duplex-bol')
if not os.path.isdir(REPO):
    # clone if it's not already sitting on disk
    subprocess.run(
        ['git', 'clone', 'https://github.com/maidah-binte-tariq/duplex-bol', REPO],
        check=False,
    )

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[moshi]'], check=True)

In [ ]:
import duplex_bol
print('duplex_bol', duplex_bol.__version__ if hasattr(duplex_bol, '__version__') else '(installed)')

## 2. Data — Common Voice Urdu

Two free Urdu sources:

- **Common Voice Urdu** (CC0): https://github.com/common-voice/cv-dataset
- **Mozilla 3-speaker Urdu set** (CC-BY-NC):
  https://mozilladatacollective.com/datasets/cmmvykcrs0050ny07vkwww5gi

On Kaggle, attach the Common Voice Urdu dataset (or download the HF mirror) so
`validated.tsv` lands on disk. Below we make a tiny placeholder TSV so the cell
runs anywhere; swap `CV_TSV` for the real path.

In [ ]:
from pathlib import Path
from duplex_bol.data import read_cv_tsv, select_speakers, count_by_speaker

# >>> download: point this at the real Common Voice Urdu validated.tsv.
# e.g. CV_TSV = Path('/kaggle/input/common-voice-urdu/ur/validated.tsv')
CV_TSV = Path('cv_demo/validated.tsv')

if not CV_TSV.exists():
    # placeholder so the notebook is runnable offline; same columns CV ships.
    CV_TSV.parent.mkdir(parents=True, exist_ok=True)
    rows = [
        ('client_a', 'a000.mp3', 'السلام علیکم', 'female_feminine'),
        ('client_a', 'a001.mp3', 'آپ کیسے ہیں', 'female_feminine'),
        ('client_a', 'a002.mp3', 'میں ٹھیک ہوں', 'female_feminine'),
        ('client_b', 'b000.mp3', 'یہ کتنے کا ہے', 'male_masculine'),
        ('client_b', 'b001.mp3', 'مجھے ایک ٹکٹ چاہیے', 'male_masculine'),
        ('client_c', 'c000.mp3', 'شکریہ بہت مہربانی', 'male_masculine'),
        ('client_c', 'c001.mp3', 'پھر ملیں گے', 'male_masculine'),
        ('client_d', 'd000.mp3', 'کیا وقت ہوا ہے', 'female_feminine'),
    ]
    with CV_TSV.open('w', encoding='utf-8') as fh:
        fh.write('client_id\tpath\tsentence\tgender\n')
        for cid, p, s, g in rows:
            fh.write(f'{cid}\t{p}\t{s}\t{g}\n')

clips = read_cv_tsv(CV_TSV)
print(f'{len(clips)} clips, {len(count_by_speaker(clips))} speakers total')

In [ ]:
# Deterministic, gender-balanced pick of 3 speakers (the report's setup).
chosen = select_speakers(clips, n=3, balance_gender=True, min_clips=1)
for sid, sel in chosen.items():
    print(f'{sid:>10}  clips={len(sel):>3}  gender={sel[0].gender_bucket}')

### 16 kHz mono WAV

Whisper wants 16 kHz mono. Common Voice ships 48 kHz MP3, so convert. The
package has the audio plumbing (`duplex_bol.audio.read_wav` / `write_wav` /
`resample` / `to_mono`), or just shell out to ffmpeg per clip:

```bash
ffmpeg -i clip.mp3 -ac 1 -ar 16000 clip.wav   # mono, 16 kHz
```

In [ ]:
# Illustrative resample helper using the package audio transforms.
# (Skips if a clip path doesn't resolve — placeholder MP3s aren't on disk.)
from duplex_bol.audio import resample, to_mono

def to_asr_wav(samples, src_sr, dst_sr=16_000):
    """mono + 16 kHz, the shape faster-whisper expects."""
    mono = to_mono(samples)
    return resample(mono, src_sr, dst_sr) if src_sr != dst_sr else mono

print('resample helper ready (mono, 16 kHz target)')

## 3. ASR eval — Urdu WER (H6)

**H6: word error rate <= ~30%** on held-out Urdu. Scoring is only fair if both
sides are normalized the same way first — ARABIC YEH vs FARSI YEH would
otherwise count as an error on every word with a yeh. So we pass
`normalize_urdu` as the normalizer.

Aggregate the *right* way: sum per-utterance edit counts, divide once.
Averaging per-utterance WERs lets one short noisy clip dominate.

In [ ]:
from duplex_bol.eval import word_error_rate, aggregate_wer
from duplex_bol.text import normalize_urdu

# (reference, hypothesis) pairs. Swap in your real ASR output.
pairs = [
    ('السلام علیکم آپ کیسے ہیں',      'سلام علیکم آپ کیسے ہیں'),     # 1 deletion
    ('میں بالکل ٹھیک ہوں شکریہ',       'میں بالکل ٹھیک ہوں شکریہ'),  # perfect
    ('مجھے ایک ٹکٹ خریدنا ہے',         'مجھے ایک ٹکٹ خریدنی ہے'),    # 1 substitution
    ('یہ کتنے پیسے کا ہے',             'یہ کتنے کا ہے'),             # 1 deletion
]

for ref, hyp in pairs:
    c = word_error_rate(ref, hyp, normalizer=normalize_urdu)
    print(f'WER {c.error_rate:5.2f}  S={c.substitutions} D={c.deletions} I={c.insertions} N={c.ref_length}')

agg = aggregate_wer(pairs, normalizer=normalize_urdu)
print(f'\ncorpus WER = {agg.error_rate:.3f}  (errors={agg.errors}, ref words={agg.ref_length})')
print('H6 (<= 0.30):', 'PASS' if agg.error_rate <= 0.30 else 'FAIL')

## 4. TTS fine-tune (GPU)

The mouth. Open Urdu voices are experimental — budget time on prosody. Two
starting points:

- **SpeechT5-Urdu**: https://huggingface.co/TheUpperCaseGuy/Guy-Urdu-TTS
- **Orpheus-Urdu**: https://huggingface.co/mahwizzzz/orpheus-urdu-tts

Fine-tune on the speaker you selected above so the agent has one consistent
voice. Sketch below (SpeechT5) — **needs a GPU**, kept illustrative.

In [ ]:
# >>> needs GPU. Illustrative SpeechT5-Urdu fine-tune skeleton.
# pip install transformers datasets accelerate soundfile
#
# from transformers import (
#     SpeechT5ForTextToSpeech, SpeechT5Processor, SpeechT5HifiGan,
#     Seq2SeqTrainer, Seq2SeqTrainingArguments,
# )
#
# processor = SpeechT5Processor.from_pretrained('TheUpperCaseGuy/Guy-Urdu-TTS')
# model     = SpeechT5ForTextToSpeech.from_pretrained('TheUpperCaseGuy/Guy-Urdu-TTS')
#
# # Build a dataset from the chosen speaker's (normalized text, 16 kHz wav) pairs.
# # Normalize text with the SAME normalizer used for WER, so train == eval spelling.
# from duplex_bol.text import normalize_urdu
# def prep(batch):
#     batch['normalized_text'] = normalize_urdu(batch['sentence'])
#     return batch
#
# args = Seq2SeqTrainingArguments(
#     output_dir='urdu-tts-spk', per_device_train_batch_size=8,
#     learning_rate=1e-4, num_train_epochs=10, fp16=True,
# )
# # Seq2SeqTrainer(model=model, args=args, train_dataset=..., ...).train()
print('TTS fine-tune is GPU-only; this cell is a reference skeleton.')

## 5. The barge-in orchestrator (runs here, on CPU)

This is the heart of Track B and it runs on free Kaggle. The orchestrator is a
pure, synchronous, frame-driven state machine. It talks to ASR / brain / TTS
only through Protocols, so here we drive it with **fakes** — what we're testing
is the *policy*, not transcription quality.

One `AudioFrame` in = one tick. We script a frame pattern: `S` = a speech frame,
`.` = silence. The caller speaks, pauses (bot replies), then talks over the bot
mid-reply — and the bot must stop fast. Mirrors the `duplex-bol demo` CLI.

In [ ]:
import numpy as np
from duplex_bol.cascade import AudioFrame, BargeIn, DuplexOrchestrator, EnergyVAD
from duplex_bol.cascade.fakes import ScriptedASR, RuleBasedAgent, ChunkedTTS
from duplex_bol.eval import LatencyBudget

def pattern_frames(pattern, frame_samples=320):
    """'S' -> a loud (speech) frame, '.' -> a silent frame."""
    speech  = np.full(frame_samples, 0.3, np.float32)
    silence = np.zeros(frame_samples, np.float32)
    return [AudioFrame(speech if c == 'S' else silence) for c in pattern]

orch = DuplexOrchestrator(
    vad=EnergyVAD(),
    asr=ScriptedASR(['السلام علیکم', 'میں ٹھیک ہوں']),
    agent=RuleBasedAgent(default='وعلیکم السلام، میں آپ کی کیا مدد کر سکتا ہوں'),
    tts=ChunkedTTS(frames_per_word=3),
)

# caller talks (SSSS), pauses (.....), bot replies and gets cut off (SSSSS...).
pattern = 'SSSS.....SSSSS.........'
for ev in orch.run(pattern_frames(pattern)):
    mark = '  <-- bot stopped' if isinstance(ev, BargeIn) else ''
    print(f'  [{ev.frame_index:>3}] {type(ev).__name__}{mark}')

In [ ]:
# Grade the run against the voice-agent budget (H4 / H5).
report = LatencyBudget.voice_agent_default().evaluate(orch.tracker)
print(report)
print('\noverall:', 'PASS' if report.ok else 'FAIL')

## 6. Acceptance criteria recap

| ID | Criterion | Where checked |
|----|-----------|---------------|
| H3 | Barge-in actually interrupts the bot | `BargeIn` event above |
| H4 | Stop latency p95 <= 500 ms | latency budget above |
| H5 | Response latency p95 <= 1000 ms | latency budget above |
| H6 | Urdu WER <= ~30% | aggregate WER above |

**Next steps**

- Swap the fakes for real adapters behind the same Protocols: faster-whisper
  (ASR), an Urdu instruct LLM (brain), Orpheus/SpeechT5-Urdu (TTS). Wire them in
  Pipecat/LiveKit — see `configs/cascade.yaml`.
- Re-run WER on real ASR output over held-out Common Voice Urdu.
- Measure barge-in / response latency on a live GPU call, not just the fakes.
- For *genuine* simultaneity (not faked), see Notebook 2 (Track A, Moshi).